# Creative Story Battle with 2 Creative Writer and Judge

In [ ]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_core import SingleThreadedAgentRuntime
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.models.ollama import OllamaChatCompletionClient
from dotenv import load_dotenv

In [ ]:
load_dotenv(override=True)

In [ ]:
@dataclass
class Message:
    content: str

In [ ]:
class Writer1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        writer_client = OllamaChatCompletionClient(model="llama3.2", temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=writer_client)

    @message_handler
    async def handle_story(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token )
        return Message(content=response.chat_message.content)

class Writer2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        writer_client = OllamaChatCompletionClient(model="llama3.2", temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=writer_client)

    @message_handler
    async def handle_story(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token )
        return Message(content=response.chat_message.content)

In [ ]:
JUDGE = "You are a judge. Evaluate stories based on creativity, coherence, and engagement. Choose the winner and justify your decision."

class StoryBattleAgent(RoutedAgent):

    def __init__(self, name: str) -> None:
        super().__init__(name)
        client = OllamaChatCompletionClient(model="llama3.2", temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=client)

    @message_handler
    async def handle_story(self, message: Message, ctx: MessageContext) -> Message:
        instruction = "You are a creative writer. Write a short, engaging story based on the given theme. Focus on originality, emotional impact, and narrative flow."
        message = Message(content=instruction)
        writer_1 = AgentId("Writer_1", "default")
        writer_2 = AgentId("Writer_2", "default")
        response1 = await self.send_message(message, writer_1)
        response2 = await self.send_message(message, writer_2)
        result = f"Writer-1 :  {response1.content}\n\nWriter-2 : {response2.content}\n"
        final_judgement = f"{JUDGE}{result} Who is the Winner? "
        message = TextMessage(content=final_judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + response.chat_message.content)

In [ ]:
runtime = SingleThreadedAgentRuntime()
await Writer1Agent.register(runtime, "Writer_1", lambda: Writer1Agent("Writer_1"))
await Writer2Agent.register(runtime, "Writer_2", lambda: Writer2Agent("Writer_2"))
await StoryBattleAgent.register(runtime, "Creative_Story_Battle", lambda: StoryBattleAgent("Creative_Story_Battle"))
runtime.start()

In [ ]:
id = AgentId("Creative_Story_Battle", "default")
message = Message(content="go")
response = await runtime.send_message(message, id)
print(response.content)

In [ ]:
await runtime.stop()
await runtime.close()